## Optional Section to bring out new exogenous non-temporal variables 

### Previous Step: Modeling (first iteration)

### Next Step: Modeling (second iteration)

---

#### TODO:

- `Analyze Client Behaviour`
- `Spatial Data Clustering`
- `Explore Correlations between old and new variables`
- `Select columns for training`

---
Based on the initial unique values for each column: 
- There have been `778` orders in total:
    + No repeated order_codes (cleaning_phase.ipynb)

- There have been `22890` different projects:
    + TODO
- There have been `2022` different clients (apparently)
- There have been `70947` different addresses for delivery

### TODO:
- Confirm that there are no clients with similar names that could be the same client (e.g., "Ruba Desarrollos" vs "Ruba Desarrollos S.A.").
- Analyze most recurrent clients and their order patterns, specially Ruba Desarrollos
- Sum least influential clients, note their build-up percentage and try to create clusters of them to analyze their behavior as a group.

### TODO:
- Check for similar project names and their relation to the same client in order to:
    + Identify client recurring patterns.
    + Clean data by unifying project names that are the same but have typos or different formats.

#### TODO:
- For the `map_page` section, create a map visualization with heat indicators to see the most common delivery locations and their distribution across the city. Also add the locations of the plants.

- IDEA: Maybe A* or Dijstra could work to design ideal routes for the trucks, based on the remissions and volume predicted. The idea could favor the final calculation of the optimal truck count for each plant.   

In [ ]:
# Categorical data distribution: Client names distribution
client_name_distribution = (
    remissions_EDA['name']
    .value_counts()
    .rename_axis('client_name')
    .reset_index(name='count')
)

client_name_distribution['percentage'] = (
    client_name_distribution['count'] / len(remissions_EDA) * 100
).round(3)

client_name_distribution.head(-1)

In [ ]:
# Categorical data distribution: Project names distribution 
project_name_distribution = (
    remissions_EDA['Nombre del proyecto']
    .value_counts()
    .rename_axis('Nombre del proyecto')
    .reset_index(name='count')
)

project_name_distribution['percentage'] = (
    project_name_distribution['count'] / len(remissions_EDA) * 100
).round(3)

project_name_distribution.head(-1)

In [ ]:
# Check for remission frequency for each project and the sum of volume per project
project_remissions = (
    remissions_cleaned.groupby('Nombre del proyecto')
    .agg({
        'tkt_code': list,
        'u_Volumen': 'sum'
    })
    .reset_index()
)
project_remissions.columns = ['project_name', 'tkt_codes', 'total_volume']
project_remissions['remission_count'] = project_remissions['tkt_codes'].str.len()
project_remissions = project_remissions.sort_values('remission_count', ascending=False).reset_index(drop=True)
project_remissions.head(-1)


In [ ]:
# Show top projects by remission count
top_n_projects = 20
fig_top_projects = plotly.express.bar(
    project_remissions.head(top_n_projects),
    x='project_name',
    y='remission_count',
    title=f'Top {top_n_projects} projects by remission count',
    text='remission_count'
)
fig_top_projects.update_traces(texttemplate='%{text}', textposition='outside', marker_color='steelblue')
fig_top_projects.update_layout(xaxis_tickangle=45)
fig_top_projects.show()

In [ ]:
# Top clients with most projects
client_projects = (
    remissions_cleaned.groupby('name')
    .agg({
        'Nombre del proyecto': pd.Series.nunique,
        'tkt_code': list
    })
    .reset_index()
)
client_projects.columns = ['client_name', 'unique_project_count', 'tkt_codes']
client_projects = client_projects.sort_values('unique_project_count', ascending=False).reset_index(drop=True)
client_projects.head(20)

In [ ]:
# Show top clients with most projects

fig_top_clients = plotly.express.bar(
    client_projects.head(20),
    x='client_name',
    y='unique_project_count',
    title='Top 20 clients by unique project count',
    text='unique_project_count'
)
fig_top_clients.update_traces(texttemplate='%{text}', textposition='outside', marker_color='seagreen')
fig_top_clients.update_layout(xaxis_tickangle=45)
fig_top_clients.show()

In [ ]:
# Categorical data distribution: map_page distribution
map_page_distribution = (
    remissions_EDA['map_page']
    .value_counts()
    .rename_axis('map_page')
    .reset_index(name='count')
)

map_page_distribution['percentage'] = (
    map_page_distribution['count'] / len(remissions_EDA) * 100
).round(2)

map_page_distribution.head(-1)

In [ ]:
# Categorical data distribution: order_code distribution
order_code_distribution = (
    remissions_EDA['order_code']
    .value_counts()
    .rename_axis('order_code')
    .reset_index(name='count')
)

order_code_distribution['percentage'] = (
    order_code_distribution['count'] / len(remissions_EDA) * 100
).round(2)

order_code_distribution.head(-1)

In [ ]:
# List repeated addresses and their frequency
address_counts = remissions_cleaned['ship_addr_line'].value_counts().reset_index()
address_counts.columns = ['ship_addr_line', 'frequency']
address_counts = address_counts.sort_values('frequency', ascending=False).reset_index(drop=True)
address_counts.head(20)

In [ ]:
# Check if lowered and stripped addresses have more duplicates, which could indicate formatting inconsistencies
remissions_cleaned['normalized_address'] = remissions_cleaned['ship_addr_line'].str.lower().str.strip()
normalized_address_counts = remissions_cleaned['normalized_address'].value_counts().reset_index()
normalized_address_counts.columns = ['normalized_address', 'frequency']
normalized_address_counts = normalized_address_counts.sort_values('frequency', ascending=False).reset_index(drop=True)

# Check if there are addresses that only differ by accents or special characters
import unidecode

remissions_cleaned['unaccented_address'] = remissions_cleaned['normalized_address'].apply(unidecode.unidecode)
unaccented_address_counts = remissions_cleaned['unaccented_address'].value_counts().reset_index()
unaccented_address_counts.columns = ['unaccented_address', 'frequency']
unaccented_address_counts = unaccented_address_counts.sort_values('frequency', ascending=False).reset_index(drop=True)
unaccented_address_counts.head(20)
#Show duplicate values in the unaccented addresses, which could indicate that some addresses are the same but have formatting differences.
unaccented_duplicates = unaccented_address_counts[unaccented_address_counts['frequency'] > 1]
unaccented_duplicates.head(20)

In [ ]:
#Check unique addresses after normalization and unaccenting vs original
unique_original_addresses = remissions_cleaned['ship_addr_line'].nunique()
unique_normalized_addresses = remissions_cleaned['normalized_address'].nunique()
unique_unaccented_addresses = remissions_cleaned['unaccented_address'].nunique()
print(f"Unique original addresses: {unique_original_addresses}")
print(f"Unique normalized addresses: {unique_normalized_addresses}")
print(f"Meaning that there are {unique_original_addresses - unique_normalized_addresses} addresses that only differ by case or leading/trailing spaces.")
print(f"Unique unaccented addresses: {unique_unaccented_addresses}")
print(f"Meaning that there are {unique_normalized_addresses - unique_unaccented_addresses} addresses that only differ by accents or special characters.")

